Ліцензовано під [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

# Винятки, тестування

## Зміст
- Винятки
    - Принцип роботи
    - Обробка винятків
    - Власні винятки
- Тестування
    - Автоматичне тестування
    - unittest

## Винятки

### Принцип роботи

Ви з винятками стикалися вже не раз під час розробки програм. Винятки переважно слугують для того, аби повідомити, що щось пішло не так, аби або пізніше обробити такі помилки (через `try-except`), або взагалі закінчити програму

Узагалі є дуже багато вбудованих у Python винятків, і, як і все в Python, вони є також об'єктами, з певною структурою успадкувань

Є предок усіх винятків - `BaseException`, від якого будуть успадковані всі інші. Від нього прямо походять такі системні винятки, як `SystemExit` чи `KeyboardInterrupt` - переважно ви не хочете обробляти ці винятки, адже вони означають, що програма закінчується

Також від нього успадковується `Exception` - батько усіх винятків, які є сенс обробляти, такі як `ValueError` чи `IndexError` (звичайно, усе трохи складніше, але базова ідея така)

Детальніше про структуру можна почитати [тут](https://docs.python.org/3/library/exceptions.html)

Після виникнення винятку, хтось має його обробити. Спочатку ця відповідальність лягає на безпосередній кусок коду, в якому він стався:

In [ ]:
def get_first(a: list[int]) -> int:
    try:
        return a[1]
    except IndexError:
        return 0


def cause_error() -> int:
    return get_first([])

print(cause_error())

0


Якщо виняток не оброблений одразу, він буде переданий вище по викликах функцій, і можна буде обробити там:

In [ ]:
def get_first(a: list[int]) -> int:
    return a[1]

def cause_error() -> int:
    try:
        return get_first([])
    except IndexError:
        return 0

print(cause_error())

0


І так аж до кінця:

In [ ]:
def get_first(a: list[int]) -> int:
    return a[1]

def cause_error() -> int:
    return get_first([])

try:
    print(cause_error())
except IndexError:
    print(0)

0


Якщо протягом усього цього шляху виняток не буде оброблений, то це зробить Python, кажучи, де і який виняток виник:

In [ ]:
def get_first(a: list[int]) -> int:
    return a[1]

def cause_error() -> int:
    return get_first([])

print(cause_error())

IndexError: list index out of range

Якщо хочемо самі спричинити буль-який виняток, можемо це зробити за допомогою `raise`:

In [ ]:
raise ValueError

ValueError: 

До речі, у цьому випадку `ValueError` є класом, а не об'єктом типу `ValueError` (сам об'єкт помилки створює `raise`). Можемо навіть це перевірити:

In [ ]:
print(type(ValueError))
print(type(ValueError()))

<class 'type'>
<class 'ValueError'>


Але також можна передавати вже готовий об'єкт помилки:

In [ ]:
raise ValueError()

ValueError: 

Або навіть давати текст помилки, аби було зрозуміліше:

In [ ]:
raise ValueError("Your ad could have been here")

ValueError: Your ad could have been here

### Обробка винятків

Нагадаємо структуру `try-except`:

In [ ]:
def example(a: list) -> int:
    to_return = 0
    try:
        to_return = a[0]
    except IndexError as e:
        print(f"An error ({e}) occured!")
        to_return = -1
    except TypeError:
        print("Another error occured!")
        to_return = -2
    else:
        print("No errors occured!")
        to_return += a[1]
    finally:
        print("This will always be printed!")
    print("This will not be always printed!")
    return to_return

print(example([1, 2]))

No errors occured!
This will always be printed!
This will not be always printed!
3


In [ ]:
print(example([]))

An error (list index out of range) occured!
This will always be printed!
This will not be always printed!
-1


In [ ]:
print(example(1))

Another error occured!
This will always be printed!
This will not be always printed!
-2


In [ ]:
print(example([1]))

No errors occured!
This will always be printed!


IndexError: list index out of range

У цьому прикладі також можемо побачити, навіщо `finally` - цей блок буде виконаний навіть якщо помилка виникне в іншому блоці (як-от `else` чи `except`). Це може бути корисно, наприклад, якщо навіть за найгірших обставин ми хочемо зробити перед закінченням програми якусь важливу дію (як-от зберегти дані, закрити з'єднання, щось вивести для користувача тощо)

Деколи ліньки роздумувати, які саме винятки можуть виникнути. Теоретично, можна написати просто `except Exception`, і таким чином обробляти будь-яку помилку, що станеться. Але це не дуже гарна практика, адже таким чином ми можемо пропустити якусь важливу помилку

Наприклад, якщо таке зробити тоді, коли насправді хочемо обробити `IndexError`, то може статися так, що ми пропустимо `KeyError`, і всіх заплутаємо!

In [ ]:
FAVOURITE_FILMS = {
    "Bohdan": ["Inside Out", "La La Land"],
    "Juliette": ["The Godfather", "Inception"],
}

def get_nth_favourite_film(person: str, index: int) -> str:
    try:
        return FAVOURITE_FILMS[person][index]
    except Exception:
        print("This person does not have that many favourite films!")
        return "Unknown"


print(get_nth_favourite_film("Bohdan", 0))
print(get_nth_favourite_film("Juliette", 2))

Inside Out
This person does not have that many favourite films!
Unknown


In [ ]:
print(get_nth_favourite_film("Juliett", 0))

This person does not have that many favourite films!
Unknown


Насправді сталася помилка, що немає такої людини, але через те, що ми обробляємо `Exception`, то навіть на цю помилку вивело "This person does not have that many favourite films"

Тому краще обирати саме ті винятки, які можуть виникнути, і обробляти їх окремо

### Власні винятки

Деколи може бути корисно мати власні винятки, аби передавати більше змісту про помилку. Ми можемо створити власний клас винятку, який успадковуватиметься від `Exception` (або від якогось конкретного винятку, якщо це має сенс):

In [ ]:
class DatabaseNotInitializedError(Exception):
    ...


def get_data() -> int:
    raise DatabaseNotInitializedError


get_data()

DatabaseNotInitializedError: 

Якщо хочемо, то можемо додавати власні атрибути до свого винятку (але бажано все ще могти приймати `message`, як і решта винятків. Для цього можемо використати `super()`):

In [ ]:
class DatabaseNotInitializedError(Exception):
    def __init__(self, message: str, database_name: str = "") -> None:
        super().__init__(message)
        self.database_name = database_name


try:
    raise DatabaseNotInitializedError("The database is not initialized!", "my_database")
except DatabaseNotInitializedError as e:
    print(f"An error occured with the database {e.database_name}: {e}")

An error occured with the database my_database: The database is not initialized!


## Тестування

### Автоматичне тестування

При розробці програми важливо перевіряти, чи вона працює правильно. Це можна робити вручну, але це:
1. Нудно
2. Можна забути перевірити якусь частину програми
3. При майбутніх змінах може зламатися те, що раніше працювало

Гарною ідеєю є написати код, що тестує код. Ми це робили із самого початку, використовуючи доктести, а потім `assert`. Однак часто це зводилося до врахування кількох входів, аби просто VPL не скаржився

Коли ж ви працюватимете над більшими програмами, то є сенс дійсно заморочуватися над тестами, які би тестували різні частини програми, особливо ті, що складні й важливі (наприклад, процес збереження даних, обробка великих файлів, авторизація, складний алгоритм тощо)

Проте коли проєкт збільшується, то виділяти більшу частину документації функцій під доктести стає незручно. Тому переважно створюють окремі файли для тестування різних частин програми (часто в окремій директорії всіх тестів)

Найочевидніший спосіб - це просто написати кілька функцій з `assert`ами

Розглянемо на простому прикладі, як це може виглядати:

In [ ]:
class Vector:
    def __init__(self, values: list[int]):
        self.values = values

    def __add__(self, other):
        if len(self.values) != len(other.values):
            raise ValueError
        return Vector([a + b for a, b in zip(self.values, other.values)])

    def __mul__(self, other):
        if len(self.values) != len(other.values):
            raise ValueError
        return sum([a * b for a, b in zip(self.values, other.values)])

In [ ]:
def test_add():
    a = Vector([1, 2, 3])
    b = Vector([4, 5, 6])
    c = a + b
    assert c.values == [5, 7, 9]

def test_add_different_lengths():
    a = Vector([1, 2, 3])
    b = Vector([4, 5])
    try:
        c = a + b
    except ValueError:
        pass
    else:
        assert False

def test_mul():
    a = Vector([1, 2, 3])
    b = Vector([4, 5, 6])
    c = a * b
    assert c == 32

def test_mul_different_lengths():
    a = Vector([1, 2, 3])
    b = Vector([4, 5])
    try:
        c = a * b
    except ValueError:
        pass
    else:
        assert False

test_add()
test_add_different_lengths()
test_mul()
test_mul_different_lengths()

Усе добре! Але було би добре, щоб якщо перший тест не спрацював, то програма не зупинялася, а написала про помилку і продовжувала тестувати інші частини програми. Гаразд, додамо ще трохи `try-except`:

In [ ]:
try:
    test_add()
except AssertionError:
    print("failed!")
try:
    test_add_different_lengths()
except AssertionError:
    print("failed!")
try:
    test_mul()
except AssertionError:
    print("failed!")
try:
    test_mul_different_lengths()
except AssertionError:
    print("failed!")

Добре, але ще гарно було б одразу вкінці одержувати звіт про те, скільки тестів відбулося, і які з них пройшли. Будемо виводити крапки щоразу, коли тест пройшов, і `F` коли не пройшов:

In [ ]:
total = 0
try:
    total += 1
    test_add()
except AssertionError:
    print("F", end="")
else:
    print(".", end="")
try:
    total += 1
    test_add_different_lengths()
except AssertionError:
    print("F", end="")
else:
    print(".", end="")
try:
    total += 1
    test_mul()
except AssertionError:
    print("F", end="")
else:
    print(".", end="")
try:
    total += 1
    test_mul_different_lengths()
except AssertionError:
    print("F", end="")
else:
    print(".", end="")
print()
print(f"{total} run, {total - 4} failed")

....
4 run, 0 failed


Щось якось ми забагато сил витратили на написання тестів. Ми просто хотіли перевірити кілька функцій, а коду-обгортки для гарного виконання й відображення результатів написали більше, ніж самих тестів!

А якщо ми захочемо додати ще якихось речей, як-от автоматичне створення вектора `a` перед кожною функцією (він же потрібний всюди однаковий, а ми його кожен раз знову створюємо) чи генерування кількох вхідних даних, то усе стане ще складніше

### unittest

`unittest` - вбудований у Python модуль, що полегшує життя при написанні тестів

Погляньмо на найпростіший приклад:

In [ ]:
import unittest

class TestVector(unittest.TestCase):
    def test_add(self):
        a = Vector([1, 2, 3])
        b = Vector([4, 5, 6])
        c = a + b
        self.assertEqual(c.values, [5, 7, 9])

    def test_add_different_lengths(self):
        a = Vector([1, 2, 3])
        b = Vector([4, 5])
        self.assertRaises(ValueError, lambda: a + b)

    def test_mul(self):
        a = Vector([1, 2, 3])
        b = Vector([4, 5, 6])
        c = a * b
        self.assertEqual(c, 32)

    def test_mul_different_lengths(self):
        a = Vector([1, 2, 3])
        b = Vector([4, 5])
        self.assertRaises(ValueError, lambda: a * b)


unittest.main(argv=[""], exit=False)

....
----------------------------------------------------------------------
Ran 4 tests in 0.002s

OK


Ми створили клас, який успадкували від `unittest.TestCase`, і написали тести у вигляді методів цього класу, а `assert`и замінили на `self.assertEqual`

Чотири крапки нагорі, так само й було в наших тестуваннях, означають, що всі чотири пройшли успішно

Це набагато простіше: нам не потрібно писати жодного `try-except` навколо тестів, рахувати кількості тощо, виводити, які саме тести не пройшли - все це робить `unittest` за нас

Також, як ви могли помітити, деякі речі `unittest` дає, що полегшують життя не тільки для виведення результатів, а й написання самих тестів. Наприклад, `assertRaises` дозволяє перевірити, чи з'являється очікуваний виняток при виклику функції

Якщо хочемо побачити більше інформації, можна додати аргумент `verbosity`:

In [ ]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_add (__main__.TestVector.test_add) ... ok
test_add_different_lengths (__main__.TestVector.test_add_different_lengths) ... ok
test_mul (__main__.TestVector.test_mul) ... ok
test_mul_different_lengths (__main__.TestVector.test_mul_different_lengths) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.005s

OK


Якщо ж якийсь тест не пройде, то виведе інформацію, який саме та чому не пройшов:

In [ ]:
class TestVector(unittest.TestCase):
    def test_add(self):
        a = Vector([1, 2, 3])
        b = Vector([4, 5, 6])
        c = a + b
        self.assertEqual(c.values, [5, 7, 9])

    def test_add_different_lengths(self):
        a = Vector([1, 2, 3])
        b = Vector([4, 5])
        self.assertRaises(ValueError, lambda: a + b)

    def test_mul(self):
        a = Vector([1, 2, 3])
        b = Vector([4, 5, 6])
        c = a * b
        self.assertEqual(c, 32)

    def test_mul_different_lengths(self):
        a = Vector([1, 2, 3])
        b = Vector([4, 5])
        self.assertRaises(ValueError, lambda: a * b)

    def test_intentional_failure(self):
        self.assertEqual(1, 2)


unittest.main(argv=[""], verbosity=2, exit=False)

test_add (__main__.TestVector.test_add) ... ok
test_add_different_lengths (__main__.TestVector.test_add_different_lengths) ... ok
test_intentional_failure (__main__.TestVector.test_intentional_failure) ... FAIL
test_mul (__main__.TestVector.test_mul) ... ok
test_mul_different_lengths (__main__.TestVector.test_mul_different_lengths) ... ok

FAIL: test_intentional_failure (__main__.TestVector.test_intentional_failure)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_23642/1336322575.py", line 25, in test_intentional_failure
    self.assertEqual(1, 2)
AssertionError: 1 != 2

----------------------------------------------------------------------
Ran 5 tests in 0.006s

FAILED (failures=1)


Також є додаткові можливості. Наприклад, можемо використати `setUp`, щоб виконати якийсь код перед кожним тестом (або `tearDown`, щоб виконати після):

In [ ]:
class TestVector(unittest.TestCase):
    def setUp(self):
        self.a = Vector([1, 2, 3])
        self.b = Vector([4, 5, 6])
        self.short_b = Vector([4, 5])

    def test_add(self):
        c = self.a + self.b
        self.assertEqual(c.values, [5, 7, 9])

    def test_add_different_lengths(self):
        self.assertRaises(ValueError, lambda: self.a + self.short_b)

    def test_mul(self):
        c = self.a * self.b
        self.assertEqual(c, 32)

    def test_mul_different_lengths(self):
        self.assertRaises(ValueError, lambda: self.a * self.short_b)


unittest.main(argv=[""], exit=False)

....
----------------------------------------------------------------------
Ran 4 tests in 0.003s

OK
